In [ ]:
from category_encoders import TargetEncoder
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import KFold, train_test_split

In [ ]:
# Wczytanie danych po etapie wranglingu
df = pd.read_pickle('samochody_clean.pkl')

print(f"Rozmiar bazy: {df.shape[0]} wierszy, {df.shape[1]} kolumn.")

In [ ]:
sns.set_theme(style="ticks")

tlumaczenia_numeryczne = {
    'price': 'Cena ofertowa (PLN)',
    'mileage': 'Przebieg (km)',
    'engine_displacement': 'Pojemność silnika (cm³)',
    'power': 'Moc silnika (KM)',
    'car_age': 'Wiek pojazdu (lata)'
}

# zmienne numeryczne do wizualizacji
zmienne_numeryczne = ['price', 'mileage', 'engine_displacement', 'power', 'car_age']

def format_duzych_liczb(x, pos):
    return "{:,}".format(int(x)).replace(',', ' ')

# Generowanie  Histogram + Boxplot dla każdej zmiennej
for col in zmienne_numeryczne:
    if col in df.columns:
        nazwa_pl = tlumaczenia_numeryczne.get(col, col)
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 4))
        
        # 1. Histogram po lewej
        sns.histplot(df[col], bins=50, kde=True, ax=axes[0], color='#4c72b0', edgecolor="white")
        axes[0].set_ylabel("Liczba pojazdów", fontsize=11)
        axes[0].set_xlabel(nazwa_pl, fontsize=11)
        axes[0].xaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
        axes[0].grid(True, linestyle='--', alpha=0.5, axis='y')
        sns.despine(ax=axes[0])
        
        # 2. Boxplot po prawej
        sns.boxplot(x=df[col], ax=axes[1], color='#55a868', flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.3})
        axes[1].set_xlabel(nazwa_pl, fontsize=11)
        axes[1].xaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
        axes[1].grid(True, linestyle='--', alpha=0.5, axis='x')
        sns.despine(ax=axes[1])
        
        plt.tight_layout()
        plt.show()

In [ ]:
sns.set_theme(style="ticks")

tlumaczenia = {
    'price': 'Cena ofertowa (PLN)',
    'mileage': 'Przebieg (km)',
    'engine_displacement': 'Pojemność silnika (cm³)',
    'power': 'Moc silnika (KM)',
    'car_age': 'Wiek pojazdu (lata)'
}

zmienne_kontynuacyjne = ['mileage', 'engine_displacement', 'power', 'car_age']

wyniki_korelacji = []

def format_duzych_liczb(x, pos):
    return "{:,}".format(int(x)).replace(',', ' ')

# Generowanie osobnego wykresu regplot dla każdej zmiennej
for col in zmienne_kontynuacyjne:
    if col in df.columns:
        nazwa_pl = tlumaczenia.get(col, col)
        
        # korelacja Spearmana
        kor = df[col].corr(df['price'], method='spearman')
        wyniki_korelacji.append({
            'Zmienna': col,
            'Opis': nazwa_pl,
            'Korelacja Spearmana z ceną': round(kor, 3)
        })

        kolor_linii = '#c44e52' if kor < 0 else '#55a868'
        
        fig, ax = plt.subplots(figsize=(8, 6))
        
        sns.regplot(
            data=df, 
            x=col, 
            y='price', 
            ax=ax,
            scatter_kws={'alpha': 0.15, 'color': '#4c72b0', 's': 10}, 
            line_kws={'color': kolor_linii, 'linewidth': 2.5}
        )
        
        ax.set_xlabel(nazwa_pl, fontsize=11)
        ax.set_ylabel(tlumaczenia['price'], fontsize=11)
        
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
        
        ax.grid(True, linestyle='--', alpha=0.5)
        sns.despine(ax=ax)
        
        plt.tight_layout()
        plt.show()

# Tworzenie i wyświetlenie tabeli podsumowującej korelacje
tabela_korelacji = pd.DataFrame(wyniki_korelacji)
display(tabela_korelacji)

In [ ]:
sns.set_theme(style="ticks")

tlumaczenia_score = {
    'score_passive_safety': 'bezpieczeństwa pasywnego',
    'score_active_safety': 'bezpieczeństwa aktywnego',
    'score_comfort': 'komfortu',
    'score_multimedia': 'multimediów',
    'score_parking': 'systemów parkowania',
    'score_lighting': 'oświetlenia',
    'score_design_interior': 'wykończenia wnętrza'
}

#  kolumny zaczynające się od score_
score_cols = [col for col in df.columns if col.startswith('score_')]

def format_duzych_liczb(x, pos):
    return "{:,}".format(int(x)).replace(',', ' ')

# Generowanie osobnego wykresu dla każdego wskaźnika wyposażenia
for col in score_cols:
    nazwa_wskaźnika = tlumaczenia_score.get(col, col)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    
    sns.countplot(
        data=df, 
        x=col, 
        ax=ax, 
        hue=col, 
        palette='crest', 
        legend=False
    )
    
    ax.set_ylabel("Liczba pojazdów", fontsize=11)
    ax.set_xlabel(f"Suma punktów wyposażenia dla wskaźnika {nazwa_wskaźnika}", fontsize=11)
    
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(format_duzych_liczb))
 
    ax.grid(True, linestyle='--', alpha=0.5, axis='y')
    sns.despine(ax=ax)
    
    plt.tight_layout()
    plt.show()

In [ ]:
sns.set_theme(style="ticks")

tlumaczenia_kategorii = {
    'seller_type_': 'Typ sprzedawcy',
    'body_type_': 'Typ nadwozia',
    'fuel_type_': 'Rodzaj paliwa',
    'state_': 'Stan pojazdu',
    'transmission_': 'Skrzynia biegów',
    'drive_type_': 'Typ napędu',
    'from_': 'Kraj pochodzenia'
}

tlumaczenia_wartosci = {

    'Gasoline': 'Benzyna',
    'Diesel': 'Diesel',
    'Gasoline + LPG': 'Benzyna + LPG',
    'Hybrid': 'Hybryda',
    'Electric': 'Elektryczny',
    
    'Manual': 'Manualna',
    'Automatic': 'Automatyczna',
    
    'Front wheels': 'Na przednie koła',
    'Rear wheels': 'Na tylne koła',
    '4x4 (attached automatically)': '4x4 (dołączany automatycznie)',
    '4x4 (fixed)': '4x4 (stały)',
    '4x4 (manually attached)': '4x4 (dołączany ręcznie)',
    
    'Private person': 'Osoba prywatna',
    'Dealer': 'Dealer',
    'Authorized Dealer': 'Autoryzowany dealer',
    
    'SUV': 'SUV',
    'Station wagon': 'Kombi',
    'Compact': 'Kompakt',
    'Sedan': 'Sedan',
    'Minivan': 'Minivan',
    'City cars': 'Auta miejskie',
    'Coupe': 'Coupe',
    'Small cars': 'Małe auta',
    'Cabriolet': 'Kabriolet',
    
    'poland': 'Polska',
    'germany': 'Niemcy',
    'france': 'Francja',
    'usa': 'USA',
    'other_abroad': 'Inny (zagranica)',
    'origin_unknown': 'Nieznane pochodzenie'
}

# Oryginalne kategorie z One-Hot Encodingu
kategorie_one_hot = ['seller_type_', 'body_type_', 'fuel_type_', 'transmission_', 'drive_type_', 'from_']

for kategoria in kategorie_one_hot:
    kolumny_grupy = [col for col in df.columns if col.startswith(kategoria)]
    
    if not kolumny_grupy:
        continue 
        
    procenty = (df[kolumny_grupy].sum() / len(df) * 100).sort_values(ascending=False)
    
    czyste_nazwy = [
        tlumaczenia_wartosci.get(nazwa.replace(kategoria, ""), nazwa.replace(kategoria, "")) 
        for nazwa in procenty.index
    ]
    
    # Korelacja Spearmana
    korelacje = []
    for col in procenty.index:
        try:
            kor = df[col].corr(df['price'], method='spearman')
            korelacje.append(kor if pd.notna(kor) else 0.0)
        except Exception:
            korelacje.append(0.0)
            
    fig, axes = plt.subplots(1, 2, figsize=(18, max(4, len(czyste_nazwy) * 0.55)))
    
 
    # Rozkład Procentowy
    ax1 = sns.barplot(
        x=procenty.values, 
        y=czyste_nazwy, 
        hue=czyste_nazwy, 
        palette='crest', 
        legend=False, 
        ax=axes[0]
    )
    
    for p in ax1.patches:
        width = p.get_width()
        if width > 0:
            ax1.text(width + 0.5, p.get_y() + p.get_height() / 2., 
                     f'{width:.1f}%', 
                     ha="left", va="center", fontsize=10, color='black')
            
    ax1.set_xlabel("Odsetek ogłoszeń (%)", fontsize=11)
    ax1.set_ylabel("") 
    ax1.set_xlim(0, min(100, procenty.max() + 15)) 

    ax1.grid(True, linestyle='--', alpha=0.5, axis='x')
    sns.despine(ax=ax1)

    # Wpływ na cenę (Korelacja Spearmana)
    kolory_wplywu = ['#c44e52' if x < 0 else '#55a868' for x in korelacje]
    
    ax2 = sns.barplot(
        x=korelacje, 
        y=czyste_nazwy, 
        hue=czyste_nazwy, 
        palette=kolory_wplywu, 
        legend=False, 
        ax=axes[1]
    )
    
    for p in ax2.patches:
        width = p.get_width()
        va_dir = "left" if width >= 0 else "right"
        offset = 0.02 if width >= 0 else -0.02
        ax2.text(width + offset, p.get_y() + p.get_height() / 2., 
                 f'{width:.2f}', 
                 ha=va_dir, va="center", fontsize=10, weight='bold', color='black')
        
    ax2.set_xlabel("Współczynnik korelacji Spearmana", fontsize=11)
    ax2.set_ylabel("") 
    
    ax2.axvline(0, color='black', linewidth=1.2, linestyle='-')
    max_abs_kor = max([abs(x) for x in korelacje] + [0.1])
    ax2.set_xlim(-max_abs_kor - 0.15, max_abs_kor + 0.15)
    
    ax2.grid(True, linestyle='--', alpha=0.5, axis='x')
    sns.despine(ax=ax2)

    plt.tight_layout()
    plt.show()

In [ ]:

# znalezienie wszystkich kolumn binarnych 
wszystkie_binarne = [
    col for col in df.columns 
    if set(df[col].dropna().unique()).issubset({0, 1})
]

# usunięcie z listy kolumny należące do grup One-Hot
kategorie_one_hot_tuple = tuple(['seller_type_', 'body_type_', 'fuel_type_', 'state_', 'transmission_', 'drive_type_', 'from_'])
niezalezne_binarne = [col for col in wszystkie_binarne if not col.startswith(kategorie_one_hot_tuple)]

procenty_binarne = (df[niezalezne_binarne].mean() * 100).sort_values(ascending=False)

# Korelacja Spearmana
korelacje_binarne = []
for col in procenty_binarne.index:
    try:
        kor = df[col].corr(df['price'], method='spearman')
        korelacje_binarne.append(kor if pd.notna(kor) else 0.0)
    except Exception:
        korelacje_binarne.append(0.0)


tlumaczenia_binarne = {
    'registered_in_poland': 'Zarejestrowany w Polsce',
    'no_accident': 'Bezwypadkowy',
    'serviced_at_aso': 'Serwisowany w ASO',
    'first_owner_from_new': 'Pierwszy właściciel',
    'damaged': 'Uszkodzony',
    'dealership_warranty_included': 'Gwarancja dealerska'
}

polskie_nazwy = [tlumaczenia_binarne.get(nazwa, nazwa) for nazwa in procenty_binarne.index]

sns.set_theme(style="ticks")

fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(niezalezne_binarne) * 0.55)))

# Rozkład Procentowy
ax1 = sns.barplot(
    x=procenty_binarne.values, 
    y=polskie_nazwy, 
    hue=polskie_nazwy, 
    palette='rocket', 
    legend=False, 
    ax=axes[0]
)

for p in ax1.patches:
    width = p.get_width()
    if width > 0:
        ax1.text(width + 0.5, p.get_y() + p.get_height() / 2., 
                 f'{width:.1f}%', 
                 ha="left", va="center", fontsize=10, color='black')

ax1.set_xlabel("Odsetek ogłoszeń w bazie (%)", fontsize=11)
ax1.set_ylabel("") 
ax1.set_xlim(0, min(100, procenty_binarne.max() + 15))

# Estetyka Modern Basic
ax1.grid(True, linestyle='--', alpha=0.5, axis='x')
sns.despine(ax=ax1)

# Wpływ na cenę (Korelacja Spearmana)
kolory_binarne = ['#c44e52' if x < 0 else '#55a868' for x in korelacje_binarne]

ax2 = sns.barplot(
    x=korelacje_binarne, 
    y=polskie_nazwy, 
    hue=polskie_nazwy, 
    palette=kolory_binarne, 
    legend=False, 
    ax=axes[1]
)

for p in ax2.patches:
    width = p.get_width()
    va_dir = "left" if width >= 0 else "right"
    offset = 0.02 if width >= 0 else -0.02
    ax2.text(width + offset, p.get_y() + p.get_height() / 2., 
             f'{width:.2f}', 
             ha=va_dir, va="center", fontsize=10, weight='bold', color='black')

ax2.set_xlabel("Współczynnik korelacji Spearmana", fontsize=11)
ax2.set_ylabel("") 

ax2.axvline(0, color='black', linewidth=1.2, linestyle='-')

max_abs_kor = max([abs(x) for x in korelacje_binarne] + [0.1])
ax2.set_xlim(-max_abs_kor - 0.15, max_abs_kor + 0.15)

ax2.grid(True, linestyle='--', alpha=0.5, axis='x')
sns.despine(ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
df_model = df.copy()

# SEPARACJA CECH I TARGETU
zmienne_do_usuniecia = ['price', 'log_price']
X = df_model.drop(columns=[col for col in zmienne_do_usuniecia if col in df_model.columns])
y = df_model['price']

# PODZIAŁ NA ZBIÓR TRENINGOWY I TESTOWY (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TWORZENIE KOPII POMOCNICZYCH
# Zapisujemy oryginalne napisy tekstowe dla wierszy, które trafiły do Train i do Test
X_train['brand_name'] = X_train['vehicle_brand']
X_train['model_name'] = X_train['vehicle_model']

X_test['brand_name'] = X_test['vehicle_brand']
X_test['model_name'] = X_test['vehicle_model']

In [ ]:
# K-FOLD TARGET ENCODING NA ZBIORZE TRENINGOWYM
X_train['encoded_brand'] = np.nan
X_train['encoded_model'] = np.nan

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]
    
    fold_encoder = TargetEncoder(cols=['vehicle_brand', 'vehicle_model'], smoothing=10.0)
    fold_encoder.fit(X_tr[['vehicle_brand', 'vehicle_model']], y_tr)
    
    
    encoded_val = fold_encoder.transform(X_val[['vehicle_brand', 'vehicle_model']])
    X_train.iloc[val_idx, X_train.columns.get_loc('encoded_brand')] = encoded_val['vehicle_brand']
    X_train.iloc[val_idx, X_train.columns.get_loc('encoded_model')] = encoded_val['vehicle_model']

X_train['vehicle_brand'] = X_train['encoded_brand']
X_train['vehicle_model'] = X_train['encoded_model']
X_train.drop(columns=['encoded_brand', 'encoded_model'], inplace=True)

In [ ]:
#  DOPASOWANIE SŁOWNIKA GLOBALNEGO I MAPOWANIE NA ZBIÓR TESTOWY
# budowa encodera na oryginalnych tekstach z całego train
global_encoder = TargetEncoder(cols=['vehicle_brand', 'vehicle_model'], smoothing=10.0)

train_texts = X_train[['brand_name', 'model_name']].rename(
    columns={'brand_name': 'vehicle_brand', 'model_name': 'vehicle_model'}
)
global_encoder.fit(train_texts, y_train)

X_test_encoded = global_encoder.transform(X_test[['vehicle_brand', 'vehicle_model']])
X_test['vehicle_brand'] = X_test_encoded['vehicle_brand']
X_test['vehicle_model'] = X_test_encoded['vehicle_model'] 

print(f"Rozmiar X_train: {X_train.shape} | Rozmiar X_test: {X_test.shape}")
print(f"Czy liczby kolumn są równe? {X_train.shape[1] == X_test.shape[1]} (Powinno być True)")

In [ ]:
# TRWAŁY ZAPIS ZBIORÓW PO PODZIALE I TARGET ENCODINGU
X_train.to_pickle('X_train.pkl')
X_test.to_pickle('X_test.pkl')
y_train.to_pickle('y_train.pkl')
y_test.to_pickle('y_test.pkl')
print("Zbiory Train i Test zostały zamrożone i przekazane do dalszych plików.")